# v6 — forma analitica, controle graduado e WDBC em baixo n

Tres coisas, nesta ordem:

**(1) Ate onde o kernel de fidelidade e' analiticamente uma gaussiana.** Com `reps=1` o kernel de
um qubit e' exatamente `cos^2(x - x')` — invariante a translacao e, para argumento pequeno,
gaussiano. Com `reps=2` **essa invariancia se perde** (o Hadamard entre camadas mistura), entao
nao ha forma fechada trivial. O que sobrevive e' o argumento de pequeno angulo: para `c` pequeno
o kernel admite expansao de segunda ordem cujo termo dominante e' quadratico no deslocamento.
A secao 1 mede o quao gaussiano cada mapa e' em funcao de `c`, e o resultado decide como a
Introducao do artigo deve ser escrita.

**(2) Controle positivo graduado.** O `ad_hoc` da v5 e' gerado com o mesmo feature map que esta
sendo testado e o surrogate cai no acaso — calibra apenas uma vantagem maxima e artificial
(+0,45). O risco real do artigo e' nao enxergar uma vantagem de 0,02 a 0,05. Aqui o parametro de
separacao e' varrido, produzindo uma familia de datasets com vantagem decrescente, e o que se
reporta e' a **menor vantagem detectavel** pelo protocolo.

**(3) WDBC em regime de baixo n**, que remove o efeito de teto da rodada anterior.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time, os
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0 * x[..., i], wires=i)
            if ent:
                for i in range(q - 1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi - x[..., i])*(np.pi - x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T) ** 2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :] - A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K - np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x)), float(r.fun)/np.sqrt(K.size)

def subsample(ytr, gtr, npc, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= npc: break
        idx += take[:npc]
    return np.array(idx)

## 1. Quao gaussiano e' o kernel, em funcao da largura de banda

Duas verificacoes. A primeira e' analitica: com `reps=1` o kernel de um qubit deve ser
exatamente `cos^2(x-x')`. A segunda mostra que com `reps=2` a invariancia a translacao se perde,
o que impede a forma fechada — e por isso o argumento do artigo precisa ser o de pequeno angulo,
nao o de fatoracao exata.

Em seguida, o residuo do melhor ajuste gaussiano por elemento da matriz de Gram, varrendo `c`.
**O ponto que decide a Introducao:** se o residuo for baixo justamente na faixa onde o
desempenho e' otimo, a coincidencia entre kernel quantico e surrogate e' esperada nesse regime.

In [ ]:
rng = np.random.default_rng(0)

# (a) forma fechada com reps=1
x1 = np.linspace(0, np.pi, 200)[:, None]
S1 = states(x1, 1, reps=1, ent=False)
K1 = gram_fid(S1, S1)
D = x1[:, 0][:, None] - x1[:, 0][None, :]
err_fechada = np.abs(K1 - np.cos(D)**2).max()

# (b) reps=2 quebra a invariancia a translacao
S2 = states(x1, 1, reps=2, ent=False)
K2 = gram_fid(S2, S2)
var_diag = max(np.diagonal(K2, k).max() - np.diagonal(K2, k).min()
               for k in range(-150, 151, 25))

painel("forma analitica", [
    f"reps=1, 1 qubit: max|K &minus; cos&sup2;(x&minus;x')| = <b>{err_fechada:.2e}</b>",
    f"reps=2, 1 qubit: variacao dentro da diagonal = <b>{var_diag:.3f}</b>"],
    destaque="Com <b>reps=1</b> o kernel e' exatamente cos&sup2; da diferenca — invariante a "
             "translacao. Com <b>reps=2</b> o Hadamard entre camadas destroi a invariancia: "
             "nao ha forma fechada em funcao de (x&minus;x'), e o argumento do artigo tem que "
             "ser o de pequeno angulo, nao o de fatoracao.")

In [ ]:
Q_ANA, N_ANA = 6, 150
Xa = rng.uniform(0, np.pi, (N_ANA, Q_ANA))
C_ANA = [0.02, 0.05, 0.07, 0.1, 0.15, 0.3, 0.6, 1.2, 2.0]

lin = []
for cv in C_ANA:
    A = Xa * cv
    row = {"c": cv}
    for nome, ent in (("produto", False), ("zz", True)):
        S = states(A, Q_ANA, 2, ent)
        K = gram_fid(S, S)
        row[f"resid_{nome}"] = round(fit_surrogate(K, A)[1], 5)
        row[f"offdiag_sd_{nome}"] = round(K[np.triu_indices_from(K, 1)].std(), 4)
    lin.append(row)
ana = pd.DataFrame(lin)

c_bom = ana.loc[ana.resid_zz.idxmin(), "c"]
painel("residuo do ajuste gaussiano por elemento", [
    f"{Q_ANA} qubits, {N_ANA} pontos uniformes, reps=2",
    "resid = &#8214;K &minus; exp(&minus;&gamma;D&sup2;)&#8214;<sub>F</sub> por elemento, "
    "com &gamma; otimo"],
    tabela=ana,
    destaque="Compare a coluna <b>resid_produto</b> com a faixa de <b>c</b> em que o AUC foi "
             "otimo nas rodadas anteriores (0,069&ndash;0,111). Se o residuo la' for da ordem de "
             "10<sup>&minus;3</sup>, o mapa de produto <b>e'</b> uma gaussiana no regime de "
             "operacao, e isso precisa ser dito na Introducao como previsao, nao como achado. "
             "O mapa ZZ tem residuo bem maior — e' o termo que rompe a aproximacao.")

## 2. Controle positivo graduado

O `gap` controla a margem imposta na geracao: valores altos produzem um problema fortemente
separavel pelo kernel quantico; valores baixos aproximam o rotulo do acaso. Varrer o `gap`
produz uma escada de vantagens, e o que interessa e' **onde o protocolo deixa de enxergar**.

Sem isso, dizer "o protocolo detecta vantagem" significa apenas que ele detecta +0,45.

In [ ]:
def gera_adhoc(n=400, dim=4, gap=0.3, seed=0):
    r = np.random.default_rng(seed)
    dev = qml.device("default.qubit", wires=dim)
    @qml.qnode(dev)
    def phi(x):
        for _ in range(2):
            for i in range(dim):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            for i in range(dim-1):
                qml.CNOT(wires=[i, i+1])
                qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                qml.CNOT(wires=[i, i+1])
        return qml.state()
    V = np.linalg.qr(r.normal(size=(2**dim,)*2) + 1j*r.normal(size=(2**dim,)*2))[0]
    D = np.diag([1.0]*(2**(dim-1)) + [-1.0]*(2**(dim-1)))
    O = V.conj().T @ D @ V
    X, y = [], []
    while len(X) < n:
        cand = r.uniform(0, 2*np.pi, (3000, dim))
        val = np.einsum('ni,ij,nj->n', (S := np.asarray(phi(cand))).conj(), O, S).real
        for xx, vv in zip(cand[np.abs(val) > gap], val[np.abs(val) > gap]):
            X.append(xx); y.append(int(vv > 0))
            if len(X) >= n: break
    return np.array(X), np.array(y)

def avalia(X, y, qubits, scales, n_folds=10, npc=None, reps=2):
    """Retorna a vantagem (auc_q - auc_surr) no otimo de bandwidth, por LOFO."""
    g = np.arange(len(y))
    npc = npc or int(np.bincount(y).min() * (n_folds-1) / n_folds)
    rows = []
    sg = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=7)
    for fold, (tr, te) in enumerate(sg.split(X, y, g)):
        r = np.random.default_rng(fold)
        sub = subsample(y[tr], g[tr], npc, r)
        ytr, yte = y[tr][sub], y[te]
        imp = SimpleImputer(strategy="median").fit(X[tr][sub])
        Xtr, Xte = imp.transform(X[tr][sub]), imp.transform(X[te])
        for q in [k for k in qubits if k <= X.shape[1]]:
            sel = SelectKBest(f_classif, k=q).fit(Xtr, ytr)
            sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
            A0 = sc.transform(sel.transform(Xtr))
            B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
            for cv in scales:
                A, B = A0*cv, B0*cv
                S1, S2 = states(A, q, reps, True), states(B, q, reps, True)
                K = gram_fid(S1, S1)
                m = SVC(kernel="precomputed").fit(K, ytr)
                gs, _ = fit_surrogate(K, A)
                ms = SVC(kernel="rbf", gamma=gs).fit(A, ytr)
                rows.append(dict(fold=fold, q=q, c=cv,
                                 auc_q=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                                 auc_s=roc_auc_score(yte, ms.decision_function(B))))
    d = pd.DataFrame(rows)
    # otimo por LOFO sobre (q, c)
    vant = []
    for f in sorted(d.fold.unique()):
        fora = d[d.fold != f].groupby(["q", "c"]).auc_q.mean()
        q_s, c_s = fora.idxmax()
        dentro = d[(d.fold == f) & (d.q == q_s) & (d.c == c_s)].iloc[0]
        vant.append(dentro.auc_q - dentro.auc_s)
    v = np.array(vant); n = len(v)
    se = v.std(ddof=1)/np.sqrt(n)
    tc = tdist.ppf(0.975, n-1)
    return d, v.mean(), v.mean()-tc*se, v.mean()+tc*se

In [ ]:
GAPS = [0.30, 0.15, 0.08, 0.04, 0.02, 0.0]
QUBITS_CTRL = [4]
SCALES = np.round(np.geomspace(0.01, 8.0, 14), 4)   # estendida: o otimo caia na borda

lin = []
t0 = time.time()
for gp in GAPS:
    X, y = gera_adhoc(n=400, dim=4, gap=gp)
    _, mu, lo, hi = avalia(X, y, QUBITS_CTRL, SCALES)
    lin.append(dict(gap=gp, prevalencia=round(y.mean(), 3), vantagem=round(mu, 4),
                    ic95_lo=round(lo, 4), ic95_hi=round(hi, 4),
                    detectada=bool(lo > 0)))
    print(f"gap={gp} ok — {time.time()-t0:.0f}s")
grad = pd.DataFrame(lin)

det = grad[grad.detectada]
menor = det.vantagem.min() if len(det) else None
nulo = grad[grad.gap == 0.0]
aviso = ""
if len(nulo) and bool(nulo.detectada.iloc[0]):
    aviso = ("<br><b>ATENCAO</b>: vantagem detectada tambem em gap=0, onde nao deveria haver "
             "estrutura. Isso indica vies do procedimento de selecao no otimo — investigue "
             "antes de usar qualquer numero deste notebook.")
painel("controle positivo graduado", [
    f"dim=4, n=400, {len(GAPS)} niveis de separacao, 10 folds",
    f"SCALES estendida ate {SCALES.max()} (na v5 o otimo caiu na borda em c=2,0)",
    f"menor vantagem com IC acima de zero: <b>{menor if menor is not None else 'nenhuma'}</b>"],
    tabela=grad,
    destaque=("Reporte no artigo: <i>o protocolo detecta vantagens a partir de "
              f"{menor if menor is not None else '[?]'} de AUC com 10 folds</i>. So entao a "
              "ausencia de vantagem nos corpora reais tem significado quantitativo." + aviso))

## 3. WDBC em regime de baixo n

Com 190 por classe o dataset satura (AUC 0,986 contra 0,984 do surrogate) e nenhuma diferenca
conseguiria aparecer. Regime escasso e' onde a literatura reivindica vantagem — e' onde a
auditoria precisa olhar.

In [ ]:
def carrega_wdbc(path="wdbc.data"):
    cols = ["id", "diag"] + [f"f{i}" for i in range(30)]
    d = pd.read_csv(path, header=None, names=cols)
    return d[[f"f{i}" for i in range(30)]].values.astype(float), (d.diag == "M").astype(int).values

Xw, yw = carrega_wdbc()
NPCS = [20, 30, 50, 100, 190]
QUBITS_W = [4, 6, 8, 10]

lin = []
t0 = time.time()
for npc in NPCS:
    d, mu, lo, hi = avalia(Xw, yw, QUBITS_W, SCALES, npc=npc)
    a = d.groupby(["q", "c"])[["auc_q", "auc_s"]].mean()
    r, p = spearmanr(a.auc_q, a.auc_s)
    lin.append(dict(n_por_classe=npc, auc_max=round(a.auc_q.max(), 4),
                    vantagem=round(mu, 4), ic95_lo=round(lo, 4), ic95_hi=round(hi, 4),
                    rho=round(r, 3), p=f"{p:.1g}"))
    print(f"npc={npc} ok — {time.time()-t0:.0f}s")
wd = pd.DataFrame(lin)

satura = wd[wd.auc_max > 0.97].n_por_classe.tolist()
usaveis = wd[wd.auc_max <= 0.97]
if len(usaveis):
    dest = (f"Regimes usaveis (AUC &lt; 0,97): {usaveis.n_por_classe.tolist()}. "
            "Reporte estes; os saturados entram apenas como nota de efeito de teto.")
else:
    dest = ("<b>Todos os regimes saturam.</b> O WDBC nao discrimina metodos neste protocolo — "
            "use-o so para mostrar que o surrogate acompanha, nunca como evidencia de ausencia "
            "de vantagem.")
painel("WDBC por tamanho de amostra",
       [f"569 amostras, 30 features", f"regimes saturados (AUC &gt; 0,97): {satura}"],
       tabela=wd, destaque=dest)

## 4. Como isso reescreve o artigo

- **Introducao.** Se a secao 1 confirmar residuo da ordem de 10<sup>-3</sup> para o mapa de
  produto na faixa otima, a coincidencia com o surrogate deixa de ser apresentada como achado e
  passa a ser *previsao* do regime de pequeno angulo. A contribuicao empirica vira: (i) o termo
  ZZ rompe essa aproximacao — residuo uma ou duas ordens maior — e (ii) romper nao compra
  desempenho.
- **Controle positivo.** Substituir "+0,4544" pela menor vantagem detectavel da secao 2.
- **WDBC.** Reportar so os regimes nao saturados.